In [156]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [157]:
from openai import OpenAI

In [158]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [159]:
import requests

In [160]:
from datetime import datetime

In [161]:
def calculate_age(birthdate):
    birth = datetime.strptime(birthdate, "%Y-%m-%d").date()
    today = datetime.today().date()

    age = today.year - birth.year

    # 아직 생일이 지나지 않았다면 1살 차감
    if (today.month, today.day) < (birth.month, birth.day):
        age -= 1

    return age

In [162]:
def convert_currency(amount):
    rate = 1330
    return amount * rate

In [163]:
def calculate_bmi(height, weight):
    height_m = height / 100
    bmi = weight / (height_m ** 2)

    return round(bmi, 2)

In [164]:
tools = [
    {
        "type" : "function",
        "name" : "calculate_age",
        "description" : "만 나이를 계산하는 함수",
        "parameters" : {
            "type" : "object",
            "properties" : {
                "birthdate" : {"type": "string"}
            },
        "required" : ["birthdate"],
        "additionalProperties" : False
        },
        "strict" : True
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "달러를 원화로 변환하는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"}
            },
            "required": ["amount"],
            "additionalProperties": False
        },
        "strict" : True
    },
    {
        "type": "function",
        "name": "calculate_bmi",
        "description": "키(cm)와 몸무게(kg)를 이용하여 BMI를 계산하는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "height": {
                    "type": "number",
                    "description": "키(cm)"
                },
                "weight": {
                    "type": "number",
                    "description": "몸무게(kg)"
                }
            },
            "required": ["height", "weight"],
            "additionalProperties": False
        },
        "strict" : True
    }
]

나이 계산 함수
예시: "1992-05-01생의 만 나이는?
함수명: calculate_age입력: birthdate(문자열, YYYY-MM-DD)

In [165]:
birthdate = input("생일을 입력해주세요(YYYY-MM-DD) : ")

user_input = f"{birthdate}생의 만 나이는?"

In [166]:
input_message = [
    {
        "role" : "user",
        "content" : user_input
    }
]

In [167]:
response = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [168]:
import json

tool_call = next(
    item for item in response.output
    if item.type == "function_call"
)

args = json.loads(tool_call.arguments)

In [169]:
result = calculate_age(args["birthdate"])

In [170]:
input_message += response.output

input_message.append(
    {
        "type" : "function_call_output",
        "call_id" : tool_call.call_id,
        "output" : str(result)
    }
)

In [171]:
response2 = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [189]:
print("=" * 50)
print("만 나이 계산 결과")
print(response2.output_text)
print("=" * 50)

만 나이 계산 결과
2002년 1월 22일생의 만 나이는 **24세**입니다.


환율 변환 함수
예시: "100달러를 원화로 바꿔줘 (환율 1330원 적용)"
함수명: convert_currency입력: amount(숫자)

In [173]:
dollar = input("원화로 바꿀 달러를 입력해주세요 : ")

user_input = f"{dollar}달러를 원화로 바꿔줘"

print("입력한 달러 : ", dollar)

입력한 달러 :  100


In [174]:
input_message = [
    {
        "role" : "user",
        "content" : user_input
    }
]

In [175]:
response = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [176]:
tool_call = next(
    item for item in response.output
    if item.type == "function_call"
)

args = json.loads(tool_call.arguments)

In [177]:
result = convert_currency(args["amount"])

In [178]:
input_message += response.output

input_message.append(
    {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    }
)

In [179]:
response3 = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [180]:
print("=" * 50)
print("환율 변환 결과")
print(response3.output_text)
print("=" * 50)

환율 변환 결과
100달러는 약 **133,000원**입니다.


BMI(체질량지수) 계산 함수
예시: "키 170cm, 몸무게 65kg의 BMI는?“
함수명: calculate_bmi입력: height(숫자, cm), weight(숫자, kg)

In [181]:
height = input("키를 입력해주세요(cm) : ")
weight = input("몸무게를 입력해주세요(kg) : ")

user_input = f"키 {height}cm, 몸무게 {weight}kg의 BMI는?"

print("입력한 키 :", height, "cm")
print("입력한 몸무게 :", weight, "kg")

입력한 키 : 170 cm
입력한 몸무게 : 66 kg


In [182]:
input_message = [
    {
        "role" : "user",
        "content" : user_input
    }
]

In [183]:
response = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [184]:
tool_call = next(
    item for item in response.output
    if item.type == "function_call"
)

args = json.loads(tool_call.arguments)

In [185]:
result = calculate_bmi(args["height"], args["weight"])

In [186]:
input_message += response.output

input_message.append(
    {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    }
)

In [187]:
response4 = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [188]:
print("=" * 40)
print("BMI 계산 결과")
print(response4.output_text)
print("=" * 40)

BMI 계산 결과
키 170cm, 몸무게 66kg의 BMI는 **22.84**입니다.
